# Method 3 — LayoutLM (text + 2D layout)

Issue #6 (Hiranya): fine-tune microsoft/layoutlm-base-uncased with word bboxes.

In [ ]:
!pip install -q transformers "datasets<3" seqeval

In [ ]:
# Make `src/` importable. In a fresh Colab runtime the repo isn't there yet, so clone it.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/chundawat-h/Receipt-Invoice-Information-Extraction.git"
if Path("../src").is_dir():            # running from notebooks/ inside the repo
    ROOT = Path("..")
elif Path("src").is_dir():
    ROOT = Path(".")
else:                                  # Colab
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    ROOT = Path("Receipt-Invoice-Information-Extraction")
sys.path.insert(0, str(ROOT.resolve()))
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

### Notes from data exploration (#4)
- `bboxes` are **already 0–1000**, so no normalisation is needed. Each sub-token gets its word's box; `[CLS]` → `[0,0,0,0]`, `[SEP]` → `[1000,1000,1000,1000]`, padding → `[0,0,0,0]`.
- BERT and LayoutLM have identical vocabularies, so both models see exactly the same sub-tokens. Use the same hyperparameters as notebook 03.
- Same truncation rule as 03: cut-off words get `"O"`. Save `results/layoutlm_preds.json` keyed by `row["key"]`.

In [ ]:
from src.data import LABEL2ID, LABELS, ID2LABEL, load_sroie

# Word-tagged SROIE (darentang/sroie, Parquet copy): train 626 / test 347.
# Columns: id, words, bboxes (already 0-1000), ner_tags, image_path, key.
# Save predictions keyed by row["key"] (e.g. "X51005230621"), not by row index.
ds = load_sroie()
ds